## [Langchain Basic RAG](https://docs.langchain.com/oss/python/langchain/knowledge-base#search-by-vector)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]="Rag-basic"

In [ ]:
# Create documents

from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [ ]:
# Generate embeddings
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model='nomic-embed-text')

In [ ]:
vector1 = embeddings.embed_query(documents[0].page_content)
vector2 = embeddings.embed_query(documents[1].page_content)

assert len(vector1) == len(vector2)
print(f"Generated vectors of length {len(vector1)}\n")
vector1[:10]

In [ ]:
# Vector store
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [ ]:
# Load a PDF
from langchain_core.documents import Document
import pypdf

def load_pdf(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page":i},
        )
        for i, page in enumerate(reader.pages)
    ]

docs = load_pdf(r"/path")
print(len(docs))
docs

In [ ]:
# Split a Document

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)

all_splits = text_splitter.split_documents(docs)

len(all_splits)
# all_splits

In [ ]:
# Index Documents
for i in range(0, len(all_splits), 20):
    batch = all_splits[i:i+20]
    ids = vector_store.add_documents(documents=batch)
    print(ids)

In [ ]:
# Search by string
results = vector_store.similarity_search(
    "Whats the query Lifecycle?"
)

print(results[0])

In [ ]:
# Async query:

results = await vector_store.asimilarity_search("Diff between core vs orm?")
print(results[0])

In [ ]:
# Return Scores
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("Whats the query Lifecycle?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

In [ ]:
# Search by vector
embedding = embeddings.embed_query("Diff between core vs orm?")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])

In [ ]:
# Use retrievers

# LangChain VectorStore objects do not subclass Runnable. 
# Retrievers are Runnables, so they support standard methods such as sync and async invoke and batch.

from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import chain

@chain
def retreiver(query: str) -> List[Document]:
    return vector_store.similarity_search(query, k=3)

queries = [
    "Diff between core vs orm?",
    "What's the query Lifecycle?",
    "What's Mapped and Mapped Column?",
    "Session vs Scalar"
]

retreiver.batch(queries)


In [ ]:
# vector Store retreivers

# retriever = vector_store.as_retriever(
#     search_type="similarity",
#     search_kwargs={"k": 1},
# )

# retriever.batch(queries)


In [ ]:
from langchain_ollama import ChatOllama
from langsmith import traceable

llm = ChatOllama(model='gemma3:4b', temperature=0.5)

@traceable
def rag_chain(question: str) -> dict:
    docs = retreiver.invoke(question)
    docs_string = "".join(doc.page_content for doc in docs)
    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.
       Use the following source documents to answer the user's questions.
       If you don't know the answer, just say that you don't know.
       Use three sentences maximum and keep the answer concise.

<context>
{docs_string}
</context>"""

    ai_msg = llm.invoke([
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    )

    return {"answer": ai_msg.content, "documents":docs, "metadata":ai_msg}

In [ ]:
question = "Diff between core vs orm? Also can you provide code example for both"

ans = rag_chain(question)

ans["answer"]